# Ejercicio 3 – Consultas directas sobre archivos Parquet

En este notebook se realiza una exploración inicial de los viajes de taxis amarillos (*yellow*) y verdes (*green*) de 2026 descargados en `data/raw/`.

Todas las consultas se ejecutan con DuckDB **directamente sobre los archivos Parquet**: no se crea ninguna tabla ni se importan datos. La conexión se abre en memoria y cada consulta indica, dentro de `FROM`, los archivos que se leen.

Las consultas se guardan como archivos `.sql` en `sql/03_exploracion/`. Cada archivo inicia con un comentario que indica su objetivo y los archivos fuente. La función `consulta()` imprime el SQL antes de ejecutarlo, de modo que cada resultado queda junto a la consulta que lo produjo.

Funciones de DuckDB utilizadas:

| Función | Uso |
|---------|-----|
| `glob(patron)` | Lista los archivos que coinciden con un patrón. |
| `parquet_file_metadata(patron)` | Lee los metadatos de cada archivo (por ejemplo, el número de filas) sin leer los datos. |
| `parquet_schema(patron)` | Lee el esquema (columnas y tipos) de cada archivo. |
| `read_parquet(patron, union_by_name = true)` | Lee los datos; `union_by_name` combina archivos con columnas distintas. |

In [1]:
import os
from pathlib import Path

import duckdb
import pandas as pd

# Se trabaja desde la raiz del proyecto para que las rutas data/raw/... funcionen
if Path.cwd().name == "notebooks":
    os.chdir("..")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DIR_SQL = Path("sql/03_exploracion")
con = duckdb.connect()  # base en memoria: no se importa ningun dato


def consulta(nombre: str) -> pd.DataFrame:
    """Muestra y ejecuta un archivo SQL de DIR_SQL; devuelve el resultado como DataFrame."""
    sql = (DIR_SQL / nombre).read_text()
    print(sql)
    return con.sql(sql).df()


print("DuckDB", duckdb.__version__)

DuckDB 1.5.5


## 3.1 Cantidad de archivos disponibles

| | |
|---|---|
| **Objetivo** | Contar los archivos Parquet por tipo de taxi y año. |
| **Fuente** | `data/raw/*/*/*.parquet` |
| **Consulta** | `sql/03_exploracion/01_cantidad_archivos.sql` |

El tipo y el año se obtienen de la ruta del archivo (`data/raw/<tipo>/<anio>/...`).

In [2]:
archivos = consulta("01_cantidad_archivos.sql")
archivos

-- Ejercicio 3.1 - Cantidad de archivos Parquet disponibles.
-- Fuente: data/raw/*/*/*.parquet (todos los tipos de taxi y anios).
-- Salida: tipo de taxi, anio y cantidad de archivos.
SELECT
    split_part(file, '/', 3) AS tipo,
    split_part(file, '/', 4) AS anio,
    count(*)                 AS archivos
FROM glob('data/raw/*/*/*.parquet')
GROUP BY ALL
ORDER BY tipo, anio;



,tipo,anio,archivos
0,green,2026,8
1,yellow,2026,8


**Resultado:** se encuentran **16 archivos**: 8 de taxis amarillos y 8 de verdes, que corresponden a enero–agosto de 2026.

**Decisión:** se confirma que la exploración cubre los mismos 8 meses para ambos tipos de taxi.

## 3.2 Cantidad de registros disponibles

| | |
|---|---|
| **Objetivo** | Contar los registros de cada archivo y el total por tipo de taxi. |
| **Fuente** | `data/raw/*/*/*.parquet` |
| **Consulta** | `sql/03_exploracion/02_cantidad_registros.sql` |

El conteo se obtiene de los **metadatos** del archivo Parquet (`num_rows`), por lo que no es necesario leer los datos.

In [3]:
registros = consulta("02_cantidad_registros.sql")
registros

-- Ejercicio 3.2 - Cantidad de registros por archivo.
-- Fuente: data/raw/*/*/*.parquet (todos los tipos de taxi y anios).
-- Salida: tipo, archivo y registros. Usa los metadatos Parquet (no lee los datos).
SELECT
    split_part(file_name, '/', 3)                  AS tipo,
    regexp_extract(file_name, '[^/]+$')            AS archivo,
    num_rows                                       AS registros
FROM parquet_file_metadata('data/raw/*/*/*.parquet')
ORDER BY tipo, archivo;



,tipo,archivo,registros
0,green,green_tripdata_2026-01.parquet,40272
1,green,green_tripdata_2026-02.parquet,37373
2,green,green_tripdata_2026-03.parquet,44208
3,green,green_tripdata_2026-04.parquet,44238
4,green,green_tripdata_2026-05.parquet,44921
5,green,green_tripdata_2026-06.parquet,44163
6,green,green_tripdata_2026-07.parquet,41252
7,green,green_tripdata_2026-08.parquet,40687
8,yellow,yellow_tripdata_2026-01.parquet,3724889
9,yellow,yellow_tripdata_2026-02.parquet,3399866


In [4]:
registros.groupby("tipo")["registros"].sum().to_frame().assign(porcentaje=lambda d: (100 * d["registros"] / d["registros"].sum()).round(2))

,registros,porcentaje
tipo,,
green,337114,1.12
yellow,29703355,98.88


**Resultado:**

- Hay **30,040,469 registros** en total: 29,703,355 de taxis amarillos (98.88 %) y 337,114 de taxis verdes (1.12 %).
- Cada mes tiene entre 3.3 y 4.1 millones de viajes amarillos y entre 37 y 45 mil viajes verdes.

**Decisión:** como los taxis amarillos son casi el 99% de los datos, las comparaciones entre ambos tipos se harán con proporciones y promedios, y no con totales absolutos.

## 3.3 Columnas presentes en los archivos

| | |
|---|---|
| **Objetivo** | Listar las columnas y contar en cuántos archivos de cada tipo aparece cada una. |
| **Fuente** | `data/raw/*/*/*.parquet` (esquema) |
| **Consulta** | `sql/03_exploracion/03_columnas.sql` |

Si una columna no aparece en los 8 archivos de un tipo, los archivos no tienen todos el mismo esquema.

In [5]:
columnas = consulta("03_columnas.sql")
columnas

-- Ejercicio 3.3 - Columnas presentes en los archivos y en cuantos archivos aparece cada una.
-- Fuente: data/raw/*/*/*.parquet (esquema Parquet de cada archivo).
-- Salida: columna, archivos yellow y archivos green en los que aparece.
SELECT
    name                                                          AS columna,
    count(*) FILTER (WHERE split_part(file_name, '/', 3) = 'yellow') AS archivos_yellow,
    count(*) FILTER (WHERE split_part(file_name, '/', 3) = 'green')  AS archivos_green
FROM parquet_schema('data/raw/*/*/*.parquet')
WHERE num_children IS NULL          -- excluye el nodo raiz del esquema
GROUP BY columna
ORDER BY archivos_yellow = 0, archivos_green = 0, columna;



,columna,archivos_yellow,archivos_green
0,DOLocationID,8,8
1,PULocationID,8,8
2,RatecodeID,8,8
3,VendorID,8,8
4,cbd_congestion_fee,8,8
5,congestion_surcharge,8,8
6,extra,8,8
7,fare_amount,8,8
8,improvement_surcharge,8,8
9,mta_tax,8,8


**Resultado:**

- **18 columnas son comunes** a ambos tipos de taxi.
- Columnas exclusivas de amarillos: `tpep_pickup_datetime`, `tpep_dropoff_datetime` y `Airport_fee`.
- Columnas exclusivas de verdes: `lpep_pickup_datetime`, `lpep_dropoff_datetime`, `ehail_fee` y `trip_type`.
- `request_source` aparece solo en **3 de 8 archivos** de cada tipo (junio, julio y agosto).

**Decisiones:**

- Para analizar ambos tipos juntos, las fechas de inicio y fin de viaje deben renombrarse a un nombre común (por ejemplo, `pickup_datetime` y `dropoff_datetime`).
- Los archivos deben leerse con `union_by_name = true`. Sin esta opción, DuckDB usa el esquema del primer archivo y `request_source` se descarta.

## 3.4 Tipos de datos de las columnas

| | |
|---|---|
| **Objetivo** | Obtener el tipo de dato de cada columna, por tipo de taxi. |
| **Fuente** | `data/raw/yellow/*/*.parquet` y `data/raw/green/*/*.parquet` |
| **Consultas** | `sql/03_exploracion/04_tipos_yellow.sql`, `sql/03_exploracion/04_tipos_green.sql` |

Ambos resultados se combinan en una sola tabla para compararlos (`NaN` indica que la columna no existe en ese tipo de taxi).

In [6]:
tipos_yellow = consulta("04_tipos_yellow.sql")
tipos_green = consulta("04_tipos_green.sql")

-- Ejercicio 3.4 - Tipos de datos de las columnas de taxis amarillos.
-- Fuente: data/raw/yellow/*/*.parquet.
-- union_by_name=true integra columnas que solo existen en algunos archivos.
DESCRIBE
SELECT *
FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true);

-- Ejercicio 3.4 - Tipos de datos de las columnas de taxis verdes.
-- Fuente: data/raw/green/*/*.parquet.
-- union_by_name=true integra columnas que solo existen en algunos archivos.
DESCRIBE
SELECT *
FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true);



In [7]:
tipos = (
    tipos_yellow[["column_name", "column_type"]]
    .merge(tipos_green[["column_name", "column_type"]], on="column_name", how="outer", suffixes=("_yellow", "_green"))
)
tipos

,column_name,column_type_yellow,column_type_green
0,Airport_fee,DOUBLE,NaN
1,DOLocationID,INTEGER,INTEGER
2,PULocationID,INTEGER,INTEGER
3,RatecodeID,BIGINT,BIGINT
4,VendorID,INTEGER,INTEGER
5,cbd_congestion_fee,DOUBLE,DOUBLE
6,congestion_surcharge,DOUBLE,DOUBLE
7,ehail_fee,NaN,DOUBLE
8,extra,DOUBLE,DOUBLE
9,fare_amount,DOUBLE,DOUBLE


**Resultado:**

- Las columnas comunes tienen **el mismo tipo** en ambos tipos de taxi:
  - fechas como `TIMESTAMP`;
  - identificadores de zona y de proveedor como `INTEGER`;
  - códigos (`RatecodeID`, `payment_type`, `passenger_count`) como `BIGINT`;
  - montos y distancia como `DOUBLE`;
  - `store_and_fwd_flag` y `request_source` como `VARCHAR`.
- `passenger_count` es `BIGINT` aunque solo representa conteos pequeños, y `RatecodeID` y `payment_type` son códigos numéricos cuyo significado se define en el diccionario de datos de la TLC.

**Decisión:** no se requieren conversiones de tipo para unir ambos conjuntos. Los códigos se interpretarán con el diccionario de datos de la TLC.

## 3.5 Muestra de registros

| | |
|---|---|
| **Objetivo** | Observar registros reales de cada tipo de taxi. |
| **Fuente** | `data/raw/yellow/*/*.parquet` y `data/raw/green/*/*.parquet` |
| **Consultas** | `sql/03_exploracion/05_muestra_yellow.sql`, `sql/03_exploracion/05_muestra_green.sql` |

Se toman 5 registros al azar con `USING SAMPLE reservoir(5 ROWS) REPEATABLE (42)`; la semilla `42` hace que la muestra sea siempre la misma.

In [8]:
consulta("05_muestra_yellow.sql")

-- Ejercicio 3.5 - Muestra aleatoria reproducible de taxis amarillos.
-- Fuente: data/raw/yellow/*/*.parquet.
SELECT *
FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true)
USING SAMPLE reservoir(5 ROWS) REPEATABLE (42);



,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,1,2026-01-01 00:34:45,2026-01-01 00:40:15,4,0.90,1,N,262,141,1,7.9,3.50,0.5,3.20,0.00,1.0,16.10,2.5,0.0,0.00,None
1,1,2026-01-01 00:06:07,2026-01-01 00:20:20,2,1.90,1,N,231,90,1,14.2,4.25,0.5,3.95,0.00,1.0,23.90,2.5,0.0,0.75,None
2,2,2026-01-01 03:15:42,2026-01-01 03:20:33,1,1.15,1,N,263,140,1,7.2,1.00,0.5,2.44,0.00,1.0,14.64,2.5,0.0,0.00,None
3,2,2026-01-01 04:25:35,2026-01-01 04:38:18,1,3.69,1,N,107,143,1,17.7,1.00,0.5,4.69,0.00,1.0,28.14,2.5,0.0,0.75,None
4,2,2026-01-01 06:33:56,2026-01-01 06:55:37,2,10.27,2,N,68,138,2,70.0,0.00,0.5,0.00,6.94,1.0,81.69,2.5,0.0,0.75,None


In [9]:
consulta("05_muestra_green.sql")

-- Ejercicio 3.5 - Muestra aleatoria reproducible de taxis verdes.
-- Fuente: data/raw/green/*/*.parquet.
SELECT *
FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true)
USING SAMPLE reservoir(5 ROWS) REPEATABLE (42);



,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,request_source
0,2,2026-01-02 05:43:34,2026-01-02 05:44:09,N,5,34,34,2,0.40,70.0,0.0,0.0,14.20,0.0,NaN,1.0,85.20,1,2,0.0,0.0,None
1,2,2026-01-02 21:02:00,2026-01-02 21:07:42,N,1,74,41,1,1.17,7.9,1.0,0.5,2.08,0.0,NaN,1.0,12.48,1,1,0.0,0.0,None
2,1,2026-01-14 21:03:03,2026-01-14 21:06:16,N,1,181,65,1,0.60,5.8,1.0,1.5,0.00,0.0,NaN,1.0,8.30,2,1,0.0,0.0,None
3,2,2026-01-16 19:11:20,2026-01-16 19:29:44,N,1,74,151,1,2.33,17.7,2.5,0.5,4.34,0.0,NaN,1.0,26.04,1,1,0.0,0.0,None
4,2,2026-01-19 02:15:45,2026-01-19 02:20:57,N,1,95,102,1,1.35,8.6,1.0,0.5,1.00,0.0,NaN,1.0,12.10,1,1,0.0,0.0,None


**Resultado:**

- Cada registro es un viaje: proveedor, fechas de inicio y fin, zonas de origen y destino (`PULocationID`, `DOLocationID`), distancia en millas, tipo de tarifa, forma de pago y el desglose del cobro (tarifa, recargos, propina, peajes y total).
- `total_amount` corresponde aproximadamente a la suma de los demás montos. Por ejemplo, en el primer viaje amarillo: 7.9 + 3.5 + 0.5 + 3.2 + 1.0 = 16.10.
- `ehail_fee` aparece vacío (`NaN`) en los viajes verdes de la muestra.
- En la muestra verde aparece un viaje de 0.40 millas y 35 segundos con tarifa de 70.0 (`RatecodeID = 5`, tarifa negociada), un caso que puede ser válido aunque es poco común.

**Decisión:** las columnas de montos se pueden analizar por componente. Las columnas vacías y los valores poco comunes se cuantifican en el siguiente paso.

## 3.6 Posibles problemas de calidad de datos

Primero se obtiene un perfil estadístico de cada columna con `SUMMARIZE`, que calcula mínimo, máximo, cantidad aproximada de valores distintos, mediana (aproximada) y porcentaje de nulos.

| | |
|---|---|
| **Objetivo** | Detectar rangos imposibles y columnas con valores faltantes. |
| **Fuente** | `data/raw/yellow/*/*.parquet` y `data/raw/green/*/*.parquet` |
| **Consultas** | `sql/03_exploracion/06_resumen_yellow.sql`, `sql/03_exploracion/06_resumen_green.sql` |

In [10]:
cols = ["column_name", "min", "max", "approx_unique", "q50", "null_percentage"]
resumen_yellow = consulta("06_resumen_yellow.sql")
resumen_yellow[cols]

-- Ejercicio 3.6 - Perfil estadistico de taxis amarillos (min, max, nulos, valores distintos).
-- Fuente: data/raw/yellow/*/*.parquet.
SUMMARIZE
SELECT *
FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true);



,column_name,min,max,approx_unique,q50,null_percentage
0,VendorID,1,7,4,2,0.00
1,tpep_pickup_datetime,2001-01-01 09:23:58,2026-08-31 23:59:59,16867928,2026-04-29 19:48:43.102882,0.00
2,tpep_dropoff_datetime,2001-01-01 16:09:38,2026-09-01 20:16:00,16587125,2026-04-30 08:20:47.84249,0.00
3,passenger_count,0,9,11,1,25.98
4,trip_distance,0.0,328522.2,7217,1.8566883927711981,0.00
5,RatecodeID,1,99,7,1,25.98
6,store_and_fwd_flag,N,Y,2,NaN,25.98
7,PULocationID,1,265,290,161,0.00
8,DOLocationID,1,265,298,162,0.00
9,payment_type,0,5,6,1,0.00


In [11]:
resumen_green = consulta("06_resumen_green.sql")
resumen_green[cols]

-- Ejercicio 3.6 - Perfil estadistico de taxis verdes (min, max, nulos, valores distintos).
-- Fuente: data/raw/green/*/*.parquet.
SUMMARIZE
SELECT *
FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true);



,column_name,min,max,approx_unique,q50,null_percentage
0,VendorID,1,6,3,2,0.00
1,lpep_pickup_datetime,2008-12-31 17:35:31,2026-08-31 23:58:28,332702,2026-05-02 22:19:16.234826,0.00
2,lpep_dropoff_datetime,2008-12-31 23:16:26,2026-09-02 09:39:37,396286,2026-05-02 22:53:35.043094,0.00
3,store_and_fwd_flag,N,Y,2,NaN,14.47
4,RatecodeID,1,99,7,1,14.47
5,PULocationID,1,265,263,75,0.00
6,DOLocationID,1,265,266,140,0.00
7,passenger_count,0,9,11,1,14.47
8,trip_distance,0.0,179830.92,2472,2.068648105030784,0.00
9,fare_amount,-500.0,1676.7,4808,13.261977790507363,0.00


**Resultado del perfil:**

| Observación | Yellow | Green |
|-------------|--------|-------|
| Fecha mínima de inicio | 2001-01-01 | 2008-12-31 |
| Fecha máxima de fin | 2026-09-01 | 2026-09-02 |
| Distancia máxima (millas) | 328,522.2 | 179,830.92 |
| `fare_amount` mínimo / máximo | -2,555.2 / 7,045.0 | -500.0 / 1,676.7 |
| `total_amount` mínimo / máximo | -2,560.2 / 7,053.5 | -501.5 / 1,678.2 |
| `RatecodeID` máximo | 99 | 99 |
| Nulos en `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge` | 25.98 % | 14.47 % |
| Nulos en `request_source` | 90.23 % | 94.30 % |
| Nulos en `ehail_fee` | — | 100 % |

Se observan fechas fuera de 2026, distancias imposibles, montos negativos y bloques de columnas con el mismo porcentaje de nulos.

A partir de estas observaciones se construye una consulta que **cuenta los registros** afectados por cada problema.

| | |
|---|---|
| **Objetivo** | Cuantificar los registros con valores sospechosos, por tipo de taxi. |
| **Fuente** | `data/raw/yellow/*/*.parquet` y `data/raw/green/*/*.parquet` |
| **Consulta** | `sql/03_exploracion/06_calidad.sql` |

La consulta une ambos tipos de taxi con `UNION ALL`, renombra `tpep_*`/`lpep_*` a `pickup`/`dropoff` y cuenta los registros de cada condición con `count(*) FILTER (WHERE ...)`. La columna `filename` permite comparar el mes del viaje con el mes del archivo.

In [12]:
calidad = consulta("06_calidad.sql")
calidad.set_index("tipo").T

-- Ejercicio 3.6 - Conteo de registros con posibles problemas de calidad por tipo de taxi.
-- Fuente: data/raw/yellow/*/*.parquet y data/raw/green/*/*.parquet.
-- Cada columna cuenta los registros que cumplen la condicion indicada en su nombre.
WITH viajes AS (
    SELECT
        'yellow'                                   AS tipo,
        filename,
        tpep_pickup_datetime                       AS pickup,
        tpep_dropoff_datetime                      AS dropoff,
        passenger_count, trip_distance, RatecodeID, payment_type,
        store_and_fwd_flag, congestion_surcharge,
        fare_amount, tip_amount, total_amount
    FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true, filename = true)
    UNION ALL
    SELECT
        'green',
        filename,
        lpep_pickup_datetime,
        lpep_dropoff_datetime,
        passenger_count, trip_distance, RatecodeID, payment_type,
        store_and_fwd_flag, congestion_surcharge,
        fare_amount, tip_amount, 

tipo,green,yellow
registros,337114,29703355
pickup_fuera_del_mes_del_archivo,98,146
dropoff_antes_de_pickup,5,10
duracion_mayor_24h,4,263
pasajeros_nulos,48775,7716688
nulos_simultaneos,48775,7716688
payment_type_nulo,48775,0
payment_type_0,0,7716688
pasajeros_cero,4527,91359
distancia_cero,12212,952231


**Resultado:**

| Problema | Yellow | % Yellow | Green | % Green |
|----------|-------:|---------:|------:|--------:|
| Inicio fuera del mes del archivo | 146 | 0.00 % | 98 | 0.03 % |
| Fin antes del inicio | 10 | 0.00 % | 5 | 0.00 % |
| Duración mayor a 24 h | 263 | 0.00 % | 4 | 0.00 % |
| Nulos simultáneos (pasajeros, tarifa, *flag*, congestión) | 7,716,688 | 25.98 % | 48,775 | 14.47 % |
| `payment_type = 0` | 7,716,688 | 25.98 % | 0 | 0.00 % |
| `payment_type` nulo | 0 | 0.00 % | 48,775 | 14.47 % |
| Pasajeros = 0 | 91,359 | 0.31 % | 4,527 | 1.34 % |
| Distancia = 0 | 952,231 | 3.21 % | 12,212 | 3.62 % |
| Distancia > 100 millas | 1,223 | 0.00 % | 72 | 0.02 % |
| `RatecodeID = 99` | 769,693 | 2.59 % | 2 | 0.00 % |
| Tarifa negativa | 157,364 | 0.53 % | 999 | 0.30 % |
| Total negativo | 161,835 | 0.54 % | 1,023 | 0.30 % |
| Total > 1,000 | 49 | 0.00 % | 1 | 0.00 % |
| Propina negativa | 883 | 0.00 % | 69 | 0.02 % |

Problemas identificados:

1. **Nulos en bloque:** los nulos de `passenger_count`, `RatecodeID`, `store_and_fwd_flag` y `congestion_surcharge` ocurren siempre en los mismos registros. En amarillos coinciden exactamente con `payment_type = 0` (*Flex Fare* en el diccionario de la TLC) y en verdes con `payment_type` nulo. Son viajes sin esa información, no errores aislados.
2. **Fechas fuera de rango:** hay viajes que inician fuera del mes del archivo (incluso en 2001 y 2008), viajes que terminan antes de iniciar y viajes de más de 24 horas.
3. **Distancias inválidas:** más del 3% de los viajes reportan distancia 0, y existen distancias mayores a 100 millas (hasta 328,522), físicamente imposibles para un viaje en taxi.
4. **Montos negativos:** cerca del 0.5% de los viajes amarillos y el 0.3% de los verdes tienen tarifa o total negativos, lo que sugiere reversos o correcciones.
5. **Códigos desconocidos:** `RatecodeID = 99` aparece en el 2.59% de los viajes amarillos.
6. **Columnas sin uso:** `ehail_fee` está vacía en el 100% de los viajes verdes y `request_source` existe solo desde junio.

**Decisiones para el análisis (Ejercicio 4):**

- Unificar ambos tipos de taxi con nombres de columna comunes y una columna `tipo`.
- Excluir los registros con inicio fuera del período del archivo, fin anterior al inicio o duración mayor a 24 h.
- Excluir distancias iguales a 0 o mayores a 100 millas y montos totales negativos.
- Conservar los viajes con nulos en bloque: son viajes válidos. Solo se excluyen en los análisis de las columnas afectadas (pasajeros, tipo de tarifa).
- No utilizar `ehail_fee` ni `request_source`.

## Resumen

- Se consultaron **16 archivos Parquet** (**30.04 millones de registros**) sin importarlos a una tabla.
- Taxis amarillos y verdes comparten 18 columnas. Las fechas de viaje tienen nombres distintos (`tpep_*` y `lpep_*`), por lo que deben renombrarse para combinarlos.
- El esquema cambia entre meses (`request_source` aparece desde junio), por lo que se debe leer con `union_by_name = true`.
- Los principales problemas de calidad son nulos en bloque asociados a la forma de pago, fechas fuera de rango, distancias de 0 o imposibles, montos negativos y códigos desconocidos.